In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc numpy
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# ضبط مستوى تحسين المحوّل

<details>
<summary><b>إصدارات الحزم</b></summary>

تم تطوير الكود في هذه الصفحة باستخدام المتطلبات التالية.
نوصي باستخدام هذه الإصدارات أو أحدث منها.

```
qiskit[all]~=2.3.0
qiskit-ibm-runtime~=0.43.1
```
</details>
الأجهزة الكمومية الحقيقية عرضة للضوضاء وأخطاء البوابات، لذا فإن تحسين الدوائر لتقليل عمقها وعدد بواباتها يمكن أن يُحسّن بشكل ملحوظ النتائج التي نحصل عليها عند تنفيذ تلك الدوائر.
تمتلك دالة [`generate_preset_pass_manager`](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.generate_preset_pass_manager#qiskit.transpiler.generate_preset_pass_manager) وسيطة موضعية مطلوبة واحدة هي `optimization_level`، وهي تتحكم في مقدار الجهد الذي يبذله المحوّل في تحسين الدوائر. يمكن أن تكون هذه الوسيطة عددًا صحيحًا يأخذ إحدى القيم 0 أو 1 أو 2 أو 3.
مستويات التحسين الأعلى تُنتج دوائر أكثر تحسينًا على حساب أوقات تصريف أطول.
يوضح الجدول التالي عمليات التحسين التي تُجرى مع كل إعداد.

<table>
  <thead>
    <tr>
      <th>مستوى التحسين</th>
      <th>الوصف</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td>0</td>
      <td>
        لا يوجد تحسين: يُستخدم عادةً لتوصيف الأجهزة
        - ترجمة أساسية
        - التخطيط/التوجيه: `TrivialLayout`، حيث يختار نفس أرقام Qubit المادية كأرقام افتراضية ويُدرج عمليات SWAP لجعلها تعمل (باستخدام `SabreSwap`)
      </td>
    </tr>
    <tr>
      <td>1</td>
      <td>
        تحسين خفيف:
        -   التخطيط/التوجيه: يُحاوَل التخطيط أولاً باستخدام `TrivialLayout`. إذا كانت هناك حاجة إلى عمليات SWAP إضافية، يُعثر على تخطيط بأقل عدد من عمليات SWAP باستخدام `SabreSwap`، ثم يستخدم `VF2LayoutPostLayout` لمحاولة اختيار أفضل Qubits في الرسم البياني.
        -   `InverseCancellation`
        -   تحسين بوابات أحادية الـ Qubit
      </td>
    </tr>
    <tr>
      <td>2</td>
      <td>
        تحسين متوسط:
          - التخطيط/التوجيه: مستوى التحسين 1 (بدون trivial) + تحسين إرشادي بعمق بحث وتجارب أكبر لدالة التحسين. نظرًا لعدم استخدام `TrivialLayout`، لا تتم محاولة استخدام نفس أرقام Qubit المادية والافتراضية.
        -   `CommutativeCancellation`
      </td>
    </tr>
    <tr>
      <td>3</td>
      <td>
        تحسين عالٍ:
        - مستوى التحسين 2 + تحسين إرشادي على التخطيط/التوجيه بجهد وتجارب أكبر
        - إعادة تركيب كتل البوابات ثنائية الـ Qubit باستخدام [تحليل KAK لكارتان](https://arxiv.org/abs/quant-ph/0507171).
        - مسارات كسر الوحدوية:
          * `OptimizeSwapBeforeMeasure`: يُعيد ترتيب القياسات لتجنب عمليات SWAP
          * `RemoveDiagonalGatesBeforeMeasure`: يُزيل البوابات قبل القياسات التي لن تؤثر على القياسات
      </td>
    </tr>
  </tbody>
</table>

## مستوى التحسين عملياً
بما أن بوابات الـ Qubit الثنائية هي عادةً المصدر الأكثر أهمية للأخطاء، يمكننا قياس "كفاءة الجهاز" للعملية التحويلية بشكل تقريبي عن طريق حساب عدد بوابات الـ Qubit الثنائية في الدائرة الناتجة.
هنا، سنجرب مستويات التحسين المختلفة على دائرة مدخلة تتكون من مؤثر وحدوي عشوائي يعقبه بوابة SWAP.

In [1]:
from qiskit import QuantumCircuit
from qiskit.circuit.library import UnitaryGate
from qiskit.quantum_info import Operator, random_unitary

UU = random_unitary(4, seed=12345)
rand_U = UnitaryGate(UU)

qc = QuantumCircuit(2)
qc.append(rand_U, range(2))
qc.swap(0, 1)
qc.draw("mpl", style="iqp")

<Image src="../docs/images/guides/set-optimization/extracted-outputs/81173ebc-8359-48a6-b585-0477907b3b93-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/set-optimization/extracted-outputs/81173ebc-8359-48a6-b585-0477907b3b93-0.svg)

سنستخدم backend المحاكى `FakeSherbrooke` في أمثلتنا. أولاً، لنُجرِّب التحويل باستخدام مستوى التحسين 0.

In [2]:
from qiskit.transpiler import generate_preset_pass_manager
from qiskit_ibm_runtime.fake_provider import FakeSherbrooke

backend = FakeSherbrooke()

pass_manager = generate_preset_pass_manager(
    optimization_level=0, backend=backend, seed_transpiler=12345
)
qc_t1_exact = pass_manager.run(qc)
qc_t1_exact.draw("mpl", idle_wires=False)

<Image src="../docs/images/guides/set-optimization/extracted-outputs/40cdd173-b437-48b1-8928-741e8411342e-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/set-optimization/extracted-outputs/40cdd173-b437-48b1-8928-741e8411342e-0.svg)

تحتوي الدائرة المحوَّلة على ست بوابات ECR ثنائية الـ Qubit.

كرر للحصول على مستوى التحسين 1:

In [3]:
from qiskit.transpiler import generate_preset_pass_manager
from qiskit_ibm_runtime.fake_provider import FakeSherbrooke

backend = FakeSherbrooke()

pass_manager = generate_preset_pass_manager(
    optimization_level=1, backend=backend, seed_transpiler=12345
)
qc_t1_exact = pass_manager.run(qc)
qc_t1_exact.draw("mpl", idle_wires=False)

<Image src="../docs/images/guides/set-optimization/extracted-outputs/2dab5def-a017-42e9-92d6-e043ac4065b2-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/set-optimization/extracted-outputs/2dab5def-a017-42e9-92d6-e043ac4065b2-0.svg)

لا تزال الدائرة المحوَّلة تحتوي على ست بوابات ECR، لكن عدد بوابات الـ Qubit الأحادية قد تقلّص.

كرر للحصول على مستوى التحسين 2:

In [4]:
pass_manager = generate_preset_pass_manager(
    optimization_level=2, backend=backend, seed_transpiler=12345
)
qc_t2_exact = pass_manager.run(qc)
qc_t2_exact.draw("mpl", idle_wires=False)

<Image src="../docs/images/guides/set-optimization/extracted-outputs/77d76048-b1e8-4225-b35f-80dc9d458e8d-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/set-optimization/extracted-outputs/77d76048-b1e8-4225-b35f-80dc9d458e8d-0.svg)

يُعطي هذا نفس النتائج كمستوى التحسين 1. لاحظ أن رفع مستوى التحسين لا يُحدث فرقاً في كل الأحوال.

كرر مجدداً، مع مستوى التحسين 3:

In [5]:
pass_manager = generate_preset_pass_manager(
    optimization_level=3, backend=backend, seed_transpiler=12345
)
qc_t3_exact = pass_manager.run(qc)
qc_t3_exact.draw("mpl", idle_wires=False)

<Image src="../docs/images/guides/set-optimization/extracted-outputs/4109d0e2-df37-4850-8409-6b860c48595c-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/set-optimization/extracted-outputs/4109d0e2-df37-4850-8409-6b860c48595c-0.svg)

الآن، لا يوجد سوى ثلاث بوابات ECR فقط. نحصل على هذه النتيجة لأنه في مستوى التحسين 3، يحاول Qiskit إعادة تركيب كتل البوابات ثنائية الـ Qubit، وأي بوابة ثنائية الـ Qubit يمكن تنفيذها باستخدام ثلاث بوابات ECR كحد أقصى. يمكننا الحصول على عدد أقل من بوابات ECR إذا ضبطنا `approximation_degree` على قيمة أقل من 1، مما يسمح للمحوّل بإجراء تقريبات قد تُدخل بعض الأخطاء في تحليل البوابات (انظر [المعاملات الشائعة الاستخدام للتحويل](common-parameters#approximation-degree)):

In [6]:
pass_manager = generate_preset_pass_manager(
    optimization_level=3,
    approximation_degree=0.99,
    backend=backend,
    seed_transpiler=12345,
)
qc_t3_approx = pass_manager.run(qc)
qc_t3_approx.draw("mpl", idle_wires=False)

<Image src="../docs/images/guides/set-optimization/extracted-outputs/bf239116-b8bb-42aa-a27a-89206d9e108a-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/set-optimization/extracted-outputs/bf239116-b8bb-42aa-a27a-89206d9e108a-0.svg)

تحتوي هذه الدائرة على بوابتي ECR فقط، لكنها دائرة تقريبية. لفهم كيف يختلف تأثيرها عن الدائرة الدقيقة، يمكننا حساب الإخلاص (fidelity) بين المؤثر الوحدوي الذي تُنفّذه هذه الدائرة والمؤثر الوحدوي الدقيق. قبل إجراء الحساب، نُقلّص أولاً الدائرة المحوَّلة التي تحتوي على 127 Qubit إلى دائرة تحتوي فقط على الـ Qubits النشطة، وعددها اثنان.

In [7]:
import numpy as np


def trace_to_fidelity_2q(trace: float) -> float:
    return (4.0 + trace * trace.conjugate()) / 20.0


# Reduce circuits down to 2 qubits so they are easy to simulate
qc_t3_exact_small = QuantumCircuit.from_instructions(qc_t3_exact)
qc_t3_approx_small = QuantumCircuit.from_instructions(qc_t3_approx)

# Compute the fidelity
exact_fid = trace_to_fidelity_2q(
    np.trace(np.dot(Operator(qc_t3_exact_small).adjoint().data, UU))
)
approx_fid = trace_to_fidelity_2q(
    np.trace(np.dot(Operator(qc_t3_approx_small).adjoint().data, UU))
)
print(
    f"Synthesis fidelity\nExact: {exact_fid:.3f}\nApproximate: {approx_fid:.3f}"
)

Synthesis fidelity
Exact: 1.000+0.000j
Approximate: 0.992+0.000j


Adjusting the optimization level can change other aspects of the circuit too, not just the number of ECR gates. For examples of how setting optimization level changes the layout, see [Representing quantum computers](./represent-quantum-computers).

## Next steps

<Admonition type="tip" title="Recommendations">
    - To learn more about the `generate_preset_passmanager` function, start with the [Transpilation default settings and configuration options](defaults-and-configuration-options) topic.
    - Continue learning about transpilation with the [Transpiler stages](transpiler-stages) topic.
    - Try the [Compare transpiler settings](/docs/guides/circuit-transpilation-settings) guide.
    - Try the [Build repetition codes](/docs/tutorials/repetition-codes) tutorial.
    - See the [Transpile API documentation.](/docs/api/qiskit/transpiler)
</Admonition>